<a href="https://colab.research.google.com/github/bangbanghy/eeg-seizure-prediction/blob/main/seizeit2_wearable_seizure_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Wearable Biosignal Processing for Seizure Prediction using SeizeIT2

## Introduction

Epilepsy(뇌전증)은 비정상적인 뇌의 전기적 활동으로 인해 반복적인 발작(seizure)이 발생하는 질환이다.

기존의 발작 탐지 및 예측 연구는 주로 EEG(Electroencephalography)를 활용해 왔다. 그러나 일상생활에서 지속적으로 EEG 장비를 착용하는 데에는 현실적인 제약이 존재한다.

최근에는 웨어러블 기기를 통해 ECG(Electrocardiography), EMG(Electromyography), Movement 등의 생체신호를 비교적 간편하게 수집할 수 있다.

본 프로젝트에서는 SeizeIT2 데이터셋의 EEG, ECG, EMG, MOV 신호를 직접 불러와 전처리하고, 향후 EEG 기반 모델과 wearable biosignal 기반 모델의 발작 탐지 성능을 비교하기 위한 데이터 분석 파이프라인을 구축하였다.

**Research Question**

EEG뿐만 아니라 ECG, EMG, Movement와 같은 wearable biosignal에서도 seizure와 관련된 정보를 추출할 수 있을까?

## Pipeline

SeizeIT2 Data → Signal Inspection → Bandpass Filtering → Normalization → Resampling → Windowing → Future LSTM Classification

현재 버전에서는 실제 생체신호의 로딩 및 전처리 파이프라인을 구현하였다.

Supervised seizure classification을 위해서는 SeizeIT2의 실제 seizure annotation을 각 signal window와 연결하는 과정이 추가적으로 필요하다.


In [ ]:
# ============================================================
# 1. Install & Import Libraries
# ============================================================

!pip -q install mne gdown

import os
import glob
import gdown
import numpy as np
import matplotlib.pyplot as plt
import mne

from scipy.signal import butter, filtfilt, resample_poly
from math import gcd

print("Libraries imported successfully.")


# 1. Load SeizeIT2 Data

본 프로젝트에서는 SeizeIT2에서 제공되는 실제 생체신호 데이터를 사용한다.

사용 신호:

- EEG: Electroencephalography
- ECG: Electrocardiography
- EMG: Electromyography
- MOV: Movement signal

각 신호는 EDF 형식으로 저장되어 있으며 MNE를 이용하여 불러온다.


In [ ]:
# ============================================================
# 2. Download and Extract Data
# ============================================================

file_id = "1MYbTW2WPmhITJiLQ-L7jJb8ZlMWRS36U"
output = "/content/sub-001_ses-01_task-szMonitoring_run-01.zip"

if not os.path.exists(output):
    gdown.download(id=file_id, output=output, quiet=False)
else:
    print("ZIP file already exists.")

extract_dir = "/content/seizeit2_data"
os.makedirs(extract_dir, exist_ok=True)

os.system(f'unzip -o "{output}" -d "{extract_dir}" > /dev/null')

print("Data extraction completed.")


In [ ]:
# ============================================================
# 3. Find EDF Files
# ============================================================

edf_files = glob.glob(
    os.path.join(extract_dir, "**", "*.edf"),
    recursive=True
)

print(f"Number of EDF files found: {len(edf_files)}")

for f in edf_files:
    print(f)


In [ ]:
# ============================================================
# 4. Automatically Match Biosignal Files
# ============================================================

def find_signal_file(files, keyword):
    matched = [
        f for f in files
        if keyword.lower() in os.path.basename(f).lower()
    ]

    if len(matched) == 0:
        raise FileNotFoundError(
            f"No EDF file containing '{keyword}' was found."
        )

    return matched[0]


eeg_path = find_signal_file(edf_files, "eeg")
ecg_path = find_signal_file(edf_files, "ecg")
emg_path = find_signal_file(edf_files, "emg")
mov_path = find_signal_file(edf_files, "mov")

print("EEG:", eeg_path)
print("ECG:", ecg_path)
print("EMG:", emg_path)
print("MOV:", mov_path)


In [ ]:
# ============================================================
# 5. Load EDF Signals
# ============================================================

eeg = mne.io.read_raw_edf(eeg_path, preload=True, verbose=False)
ecg = mne.io.read_raw_edf(ecg_path, preload=True, verbose=False)
emg = mne.io.read_raw_edf(emg_path, preload=True, verbose=False)
mov = mne.io.read_raw_edf(mov_path, preload=True, verbose=False)

print("All signals loaded successfully.")

print("\nSampling frequencies")
print("EEG:", eeg.info["sfreq"], "Hz")
print("ECG:", ecg.info["sfreq"], "Hz")
print("EMG:", emg.info["sfreq"], "Hz")
print("MOV:", mov.info["sfreq"], "Hz")

print("\nNumber of channels")
print("EEG:", len(eeg.ch_names))
print("ECG:", len(ecg.ch_names))
print("EMG:", len(emg.ch_names))
print("MOV:", len(mov.ch_names))


# 2. Exploratory Signal Analysis

모델링에 앞서 각 생체신호의 sampling frequency와 waveform을 확인한다.

신호마다 sampling frequency가 다르기 때문에 이후 동일한 sampling rate로 resampling하여 시간축을 통일한다.


In [ ]:
# ============================================================
# 6. Extract First Channel
# ============================================================

eeg_signal = eeg.get_data()[0]
ecg_signal = ecg.get_data()[0]
emg_signal = emg.get_data()[0]
mov_signal = mov.get_data()[0]

fs_eeg = int(eeg.info["sfreq"])
fs_ecg = int(ecg.info["sfreq"])
fs_emg = int(emg.info["sfreq"])
fs_mov = int(mov.info["sfreq"])

print("Signal lengths")
print("EEG:", len(eeg_signal))
print("ECG:", len(ecg_signal))
print("EMG:", len(emg_signal))
print("MOV:", len(mov_signal))


In [ ]:
# ============================================================
# 7. Visualize Raw Signals
# ============================================================

signals = [
    ("EEG", eeg_signal, fs_eeg),
    ("ECG", ecg_signal, fs_ecg),
    ("EMG", emg_signal, fs_emg),
    ("MOV", mov_signal, fs_mov)
]

for name, signal, fs in signals:
    duration = min(5, len(signal) / fs)
    n_samples = int(duration * fs)
    time = np.arange(n_samples) / fs

    plt.figure(figsize=(12, 3))
    plt.plot(time, signal[:n_samples])
    plt.title(f"Raw {name} Signal")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.show()


# 3. Signal Preprocessing

생체신호에는 baseline drift, motion artifact 및 고주파 noise 등이 포함될 수 있다.

따라서 다음 전처리를 수행한다.

1. Bandpass filtering
2. Z-score normalization
3. Sampling rate 통일
4. Signal length 정렬

필터의 상한 주파수는 각 신호의 Nyquist frequency를 초과하지 않도록 자동으로 조정한다.


In [ ]:
# ============================================================
# 8. Bandpass Filtering
# ============================================================

def bandpass_filter(signal, lowcut, highcut, fs, order=4):
    nyquist = fs / 2
    highcut = min(highcut, nyquist * 0.95)

    if lowcut >= highcut:
        raise ValueError(
            f"Invalid filter range: {lowcut}-{highcut} Hz for fs={fs}"
        )

    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(
        order,
        [low, high],
        btype="band"
    )

    return filtfilt(b, a, signal)


def zscore_normalize(signal):
    std = np.std(signal)

    if std == 0:
        return signal - np.mean(signal)

    return (signal - np.mean(signal)) / std


eeg_f = bandpass_filter(
    eeg_signal,
    lowcut=0.5,
    highcut=45,
    fs=fs_eeg
)

ecg_f = bandpass_filter(
    ecg_signal,
    lowcut=0.5,
    highcut=45,
    fs=fs_ecg
)

emg_f = bandpass_filter(
    emg_signal,
    lowcut=20,
    highcut=450,
    fs=fs_emg
)

mov_f = bandpass_filter(
    mov_signal,
    lowcut=0.1,
    highcut=20,
    fs=fs_mov
)

eeg_f = zscore_normalize(eeg_f)
ecg_f = zscore_normalize(ecg_f)
emg_f = zscore_normalize(emg_f)
mov_f = zscore_normalize(mov_f)

print("Filtering and normalization completed.")


In [ ]:
# ============================================================
# 9. Visualize Filtered Signals
# ============================================================

filtered_signals = [
    ("EEG", eeg_f, fs_eeg),
    ("ECG", ecg_f, fs_ecg),
    ("EMG", emg_f, fs_emg),
    ("MOV", mov_f, fs_mov)
]

for name, signal, fs in filtered_signals:
    duration = min(5, len(signal) / fs)
    n_samples = int(duration * fs)
    time = np.arange(n_samples) / fs

    plt.figure(figsize=(12, 3))
    plt.plot(time, signal[:n_samples])
    plt.title(f"Filtered & Normalized {name}")
    plt.xlabel("Time (s)")
    plt.ylabel("Normalized amplitude")
    plt.show()


# 4. Resampling

EEG, ECG, EMG, MOV 신호의 sampling frequency가 서로 다르기 때문에 모든 신호를 **100 Hz**로 통일한다.

동일한 sampling rate를 사용하면 이후 동일한 시간 window를 기준으로 서로 다른 생체신호를 비교할 수 있다.


In [ ]:
# ============================================================
# 10. Resampling
# ============================================================

FS_TARGET = 100


def resample_signal(signal, original_fs, target_fs):
    g = gcd(int(original_fs), int(target_fs))

    up = int(target_fs // g)
    down = int(original_fs // g)

    return resample_poly(signal, up, down)


eeg_r = resample_signal(eeg_f, fs_eeg, FS_TARGET)
ecg_r = resample_signal(ecg_f, fs_ecg, FS_TARGET)
emg_r = resample_signal(emg_f, fs_emg, FS_TARGET)
mov_r = resample_signal(mov_f, fs_mov, FS_TARGET)

min_len = min(
    len(eeg_r),
    len(ecg_r),
    len(emg_r),
    len(mov_r)
)

eeg_r = eeg_r[:min_len]
ecg_r = ecg_r[:min_len]
emg_r = emg_r[:min_len]
mov_r = mov_r[:min_len]

print("Resampling completed.")
print("Target sampling frequency:", FS_TARGET, "Hz")

print("\nSignal lengths")
print("EEG:", len(eeg_r))
print("ECG:", len(ecg_r))
print("EMG:", len(emg_r))
print("MOV:", len(mov_r))


In [ ]:
# ============================================================
# 11. Construct Wearable Multichannel Signal
# ============================================================

wearable = np.stack(
    [
        ecg_r,
        emg_r,
        mov_r
    ],
    axis=1
)

print("EEG shape:", eeg_r.shape)
print("Wearable shape:", wearable.shape)

print("\nWearable channels:")
print("0 = ECG")
print("1 = EMG")
print("2 = MOV")


# 5. Window Generation

LSTM과 같은 sequence model에 입력하기 위해 연속적인 생체신호를 일정한 길이의 window로 분할한다.

본 예시에서는 **5초 window**를 사용한다.

Sampling rate가 100 Hz이므로 하나의 window에는 500개의 time point가 포함된다.

현재 단계에서는 seizure annotation을 임의로 생성하지 않는다.

따라서 아래 과정은 **모델 입력 데이터 X만 생성**하며, 실제 seizure/non-seizure label `y`는 생성하지 않는다.


In [ ]:
# ============================================================
# 12. Generate Signal Windows
# ============================================================

WINDOW_SEC = 5
WINDOW_SIZE = FS_TARGET * WINDOW_SEC


def make_eeg_windows(signal, window_size):
    windows = []

    for start in range(
        0,
        len(signal) - window_size + 1,
        window_size
    ):
        end = start + window_size
        windows.append(signal[start:end])

    return np.asarray(
        windows,
        dtype=np.float32
    )


def make_wearable_windows(signal, window_size):
    windows = []

    for start in range(
        0,
        len(signal) - window_size + 1,
        window_size
    ):
        end = start + window_size
        windows.append(signal[start:end, :])

    return np.asarray(
        windows,
        dtype=np.float32
    )


X_eeg = make_eeg_windows(
    eeg_r,
    WINDOW_SIZE
)

X_wearable = make_wearable_windows(
    wearable,
    WINDOW_SIZE
)

print("EEG windows:", X_eeg.shape)
print("Wearable windows:", X_wearable.shape)


In [ ]:
# ============================================================
# 13. Inspect Model-ready Windows
# ============================================================

print("===== EEG =====")
print("Number of windows:", X_eeg.shape[0])
print("Time points per window:", X_eeg.shape[1])

print("\n===== Wearable =====")
print("Number of windows:", X_wearable.shape[0])
print("Time points per window:", X_wearable.shape[1])
print("Number of channels:", X_wearable.shape[2])

time = np.arange(WINDOW_SIZE) / FS_TARGET

plt.figure(figsize=(12, 3))
plt.plot(time, X_eeg[0])
plt.title("Example 5-second EEG Window")
plt.xlabel("Time (s)")
plt.ylabel("Normalized amplitude")
plt.show()


# 6. Current Results & Future Work

## Current Results

본 프로젝트에서는 실제 SeizeIT2 생체신호 데이터를 이용하여 다음 분석 파이프라인을 구현하였다.

1. SeizeIT2 EDF 데이터 로딩
2. EEG / ECG / EMG / MOV 신호 추출
3. 신호별 sampling frequency 확인
4. Bandpass filtering
5. Z-score normalization
6. 100 Hz resampling
7. EEG와 wearable biosignal의 시간축 정렬
8. ECG / EMG / MOV 기반 multichannel wearable representation 생성
9. 5초 단위 sequence window 생성

이를 통해 EEG 기반 모델과 wearable biosignal 기반 모델에 입력할 수 있는 데이터 구조를 구축하였다.

---

## Important Limitation

현재 notebook에서는 실제 seizure annotation과 signal window의 시간 정보를 연결하지 않았기 때문에 **seizure classification accuracy를 계산하지 않았다.**

신호의 평균값이나 random label을 이용해 seizure/non-seizure label을 임의로 생성하면 실제 모델 성능으로 해석할 수 없기 때문이다.

따라서 현재 결과는 seizure prediction model 자체의 성능이 아니라 **모델링을 위한 signal preprocessing pipeline 구축 결과**로 해석해야 한다.

---

## Future Work

다음 단계에서는 SeizeIT2에서 제공되는 실제 seizure annotation을 이용하여 각 window를 다음과 같이 labeling할 예정이다.

- Interictal: seizure와 관련되지 않은 구간
- Preictal: seizure 발생 이전 구간
- Ictal: seizure가 발생한 구간

이후 동일한 annotation을 이용하여 다음 두 모델을 학습하고 비교할 수 있다.

### Model 1

EEG → LSTM → Seizure Classification

### Model 2

ECG + EMG + MOV → Multivariate LSTM → Seizure Classification

평가 지표:

- Sensitivity
- Specificity
- Precision
- Recall
- F1-score

최종적으로 EEG 기반 모델과 wearable biosignal 기반 모델의 성능을 비교하여 wearable signal을 이용한 seizure monitoring의 가능성을 평가할 수 있다.
